# Vehicle Detection Deployment & Inference

This notebook provides a unified inference pipeline for 4 vehicle detection models:
- **YOLOv8**: Fast, end-to-end detection
- **SSD**: Single-shot multi-box detector
- **FasterRCNN**: Two-stage detector with region proposals
- **HOG+SVM**: Classical sliding window with hand-crafted features

Each model has been trained on the same vehicle detection dataset with classes: Bus, Car, Motorbike, Truck.
This notebook unifies their inference APIs for consistent prediction and easy comparison.

In [1]:
%%capture
!pip install -q ultralytics gradio albumentations scikit-learn joblib torch torchvision

In [2]:
import os
import cv2
import time
import torch
import joblib
import gradio as gr
import numpy as np
from pathlib import Path
from collections import defaultdict

import albumentations as A
from PIL import Image
from ultralytics import YOLO

from torchvision.models.detection import fasterrcnn_resnet50_fpn, ssd300_vgg16
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
except:
    IS_COLAB = False

print(f"Running in Colab: {IS_COLAB}")

Mounted at /content/drive
Running in Colab: True


In [4]:
# ============================================================================
# CONFIG: Model Paths and Hyperparameters
# ============================================================================

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Class mapping (all models use the same 4 classes)
CLASSES = ['Bus', 'Car', 'Motorbike', 'Truck']
IDX2CLASS = {i+1: c for i, c in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES) + 1  # +1 for background

# Model paths
if IS_COLAB:
    MODELS_PATH = '/content/drive/MyDrive/CS231/CS231_VehicleDetection/models'
else:
    MODELS_PATH = Path('../models')  # Local fallback

YOLO_MODEL_PATH = f'{MODELS_PATH}/yolov8.pt'
SSD_MODEL_PATH = f'{MODELS_PATH}/best_SSD.pth'
FASTERRCNN_MODEL_PATH = f'{MODELS_PATH}/best_FasterRCNN.pth'
HOG_SVM_MODEL_PATH = f'{MODELS_PATH}/best_hog_svm.pkl'

# Confidence thresholds
CONF_THRESHOLD = 0.5
SCORE_THRESHOLD = 0.7

print(f"Device: {DEVICE}")
print(f"Models directory: {MODELS_PATH}")
print(f"Classes: {CLASSES}")

Device: cpu
Models directory: /content/drive/MyDrive/CS231/CS231_VehicleDetection/models
Classes: ['Bus', 'Car', 'Motorbike', 'Truck']


In [5]:
# ============================================================================
# SECTION 1: Model Loading Functions
# ============================================================================

def load_yolo_model():
    """Load YOLOv8 model from pretrained weights."""
    print("Loading YOLOv8 model...")
    model = YOLO(YOLO_MODEL_PATH)
    model.to(DEVICE)
    return model

def load_ssd_model():
    """Load SSD300 model with custom classification head."""
    print("Loading SSD300 model...")
    model = ssd300_vgg16(weights=None)

    # Prepare classification head
    in_channels = []
    for layer in model.head.classification_head.module_list:
        in_channels.append(layer.in_channels)

    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = SSDClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES
    )

    # Load pretrained weights
    model.load_state_dict(torch.load(SSD_MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model

def load_fasterrcnn_model():
    """Load FasterRCNN model with custom box predictor."""
    print("Loading FasterRCNN model...")
    model = fasterrcnn_resnet50_fpn(weights=None)

    # Customize box predictor
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, NUM_CLASSES)

    # Load pretrained weights
    model.load_state_dict(torch.load(FASTERRCNN_MODEL_PATH, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    return model

def load_hog_svm_model():
    """Load HOG+SVM artifacts (scaler, classifier, config)."""
    print("Loading HOG+SVM model...")
    artifacts = joblib.load(HOG_SVM_MODEL_PATH)
    return artifacts

print("Model loading functions defined.")

Model loading functions defined.


In [6]:
# ============================================================================
# SECTION 2: Preprocessing & Helper Functions
# ============================================================================

# Albumentations transform for SSD/other models
def get_ssd_transform(size=300):
    """Preprocessing pipeline for SSD300."""
    return A.Compose([
        A.Resize(size, size),
        A.Normalize()
    ])

# HOG+SVM specific functions
def compute_iou(box_a, box_b):
    """Compute Intersection over Union for two bounding boxes."""
    x1_a, y1_a, x2_a, y2_a = box_a
    x1_b, y1_b, x2_b, y2_b = box_b

    # Intersection
    ix1 = max(x1_a, x1_b)
    iy1 = max(y1_a, y1_b)
    ix2 = min(x2_a, x2_b)
    iy2 = min(y2_a, y2_b)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih

    # Union
    area_a = max(0.0, x2_a - x1_a) * max(0.0, y2_a - y1_a)
    area_b = max(0.0, x2_b - x1_b) * max(0.0, y2_b - y1_b)
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0

def nms_boxes(boxes, scores, iou_threshold=0.4):
    """Non-Maximum Suppression to remove overlapping detections."""
    if len(boxes) == 0:
        return []

    boxes = np.asarray(boxes, dtype=np.float32)
    scores = np.asarray(scores, dtype=np.float32)

    # Sort by score (descending)
    order = scores.argsort()[::-1]
    keep = []

    while order.size > 0:
        i = order[0]
        keep.append(i)

        if order.size == 1:
            break

        rest = order[1:]
        ious = np.array([compute_iou(boxes[i], boxes[j]) for j in rest], dtype=np.float32)
        order = rest[ious < iou_threshold]

    return keep

print("Helper functions defined.")

Helper functions defined.


In [7]:
# ============================================================================
# SECTION 3: Unified Prediction Functions
# ============================================================================

# --- YOLOv8 Inference ---
def predict_yolo(model, image_rgb):
    """
    YOLOv8 Prediction Pipeline

    Input Format: RGB numpy array (H x W x 3)
    Processing: Direct to model (handles BGR conversion internally)
    Output: Detections with bounding boxes and class IDs (0-indexed)
    """
    # YOLOv8 expects BGR format
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    with torch.no_grad():
        results = model(image_bgr, conf=CONF_THRESHOLD, verbose=False)

    detections = []
    result = results[0]

    if result.boxes is not None:
        for box, cls_id, conf in zip(result.boxes.xyxy, result.boxes.cls, result.boxes.conf):
            x1, y1, x2, y2 = box.cpu().numpy()
            detections.append({
                'bbox': [float(x1), float(y1), float(x2), float(y2)],
                'class_id': int(cls_id.item()),
                'class_name': CLASSES[int(cls_id.item())],
                'confidence': float(conf.item())
            })

    return detections

# --- SSD300 Inference ---
def predict_ssd(model, image_rgb):
    """
    SSD300 Prediction Pipeline

    Input Format: RGB numpy array (H x W x 3)
    Preprocessing: Resize to 300x300, normalize
    Output: Scaled bounding boxes and 1-indexed class IDs
    """
    h_orig, w_orig = image_rgb.shape[:2]
    transform = get_ssd_transform(300)

    # Preprocess
    transformed = transform(image=image_rgb)
    img_tensor = torch.tensor(transformed['image'], dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    # Inference
    with torch.no_grad():
        outputs = model([img_tensor.squeeze(0)])[0]

    detections = []

    if outputs['boxes'] is not None and len(outputs['boxes']) > 0:
        for box, label, score in zip(outputs['boxes'], outputs['labels'], outputs['scores']):
            score = float(score.item())
            if score < CONF_THRESHOLD:
                continue

            # Scale coordinates back to original image size
            x1, y1, x2, y2 = box.cpu().numpy()
            x1 = int(x1 * w_orig / 300)
            y1 = int(y1 * h_orig / 300)
            x2 = int(x2 * w_orig / 300)
            y2 = int(y2 * h_orig / 300)

            class_id = int(label.item()) - 1  # Convert from 1-indexed to 0-indexed

            detections.append({
                'bbox': [float(x1), float(y1), float(x2), float(y2)],
                'class_id': class_id,
                'class_name': CLASSES[class_id] if 0 <= class_id < len(CLASSES) else 'Unknown',
                'confidence': score
            })

    return detections

# --- FasterRCNN Inference ---
def predict_fasterrcnn(model, image_rgb):
    """
    FasterRCNN Prediction Pipeline

    Input Format: RGB numpy array (H x W x 3)
    Preprocessing: Resize to 640x640, normalize (0-1)
    Output: Scaled bounding boxes and 1-indexed class IDs
    """
    h_orig, w_orig = image_rgb.shape[:2]

    # Preprocess
    img_resized = cv2.resize(image_rgb, (640, 640))
    img_tensor = torch.tensor(img_resized / 255., dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    # Inference
    with torch.no_grad():
        outputs = model([img_tensor.squeeze(0)])[0]

    detections = []

    if outputs['boxes'] is not None and len(outputs['boxes']) > 0:
        for box, label, score in zip(outputs['boxes'], outputs['labels'], outputs['scores']):
            score = float(score.item())
            if score < CONF_THRESHOLD:
                continue

            # Scale coordinates back to original image size
            x1, y1, x2, y2 = box.cpu().numpy()
            x1 = int(x1 * w_orig / 640)
            y1 = int(y1 * h_orig / 640)
            x2 = int(x2 * w_orig / 640)
            y2 = int(y2 * h_orig / 640)

            class_id = int(label.item()) - 1  # Convert from 1-indexed to 0-indexed

            detections.append({
                'bbox': [float(x1), float(y1), float(x2), float(y2)],
                'class_id': class_id,
                'class_name': CLASSES[class_id] if 0 <= class_id < len(CLASSES) else 'Unknown',
                'confidence': score
            })

    return detections

print("Prediction functions defined.")

Prediction functions defined.


In [8]:
# --- HOG+SVM Inference ---
def predict_hog_svm(artifacts, image_rgb):
    """
    HOG+SVM Prediction Pipeline

    Input Format: RGB numpy array (H x W x 3)
    Preprocessing: Sliding window with multiple scales, HOG feature extraction
    Output: Detected boxes with NMS filtering
    """
    scaler = artifacts['scaler']
    clf = artifacts['clf']
    config = artifacts['hyperparameters']

    # Configuration
    PATCH_SIZE = config['PATCH_SIZE']
    WINDOW_STRIDE = 32
    Y_SEARCH_RANGE = (0.2, 0.90)
    NMS_IOU_THRESHOLD = 0.35

    # Estimate scales from object sizes in dataset
    def estimate_scales(patch_size, data_range=1.5):
        return [0.65, 0.8854166865348816, 1.3958333730697632, 2.0625, 2.6]


    WINDOW_SCALES = estimate_scales(PATCH_SIZE)

    # Initialize HOG descriptor
    cell_size = config['PIX_PER_CELL']
    cell_block = config['CELLS_PER_BLOCK']
    block_size = (cell_size[0] * cell_block[0], cell_size[1] * cell_block[1])
    block_stride = (cell_size[0], cell_size[1])

    HOG = cv2.HOGDescriptor(
        PATCH_SIZE, block_size, block_stride, cell_size, config['HOG_BINS']
    )

    # Convert to BGR for cv2
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    h, w = image_bgr.shape[:2]

    # Generate sliding windows
    def sliding_windows(img_w, img_h, scales, stride):
        ys = int(img_h * Y_SEARCH_RANGE[0])
        ye = int(img_h * Y_SEARCH_RANGE[1])
        windows = []
        for s in scales:
            bw = int(PATCH_SIZE[0] * s)
            bh = int(PATCH_SIZE[1] * s)
            if bw >= img_w or bh >= img_h:
                continue
            y_stop = max(ys + 1, ye - bh)
            for y in range(ys, y_stop, stride):
                for x in range(0, img_w - bw, stride):
                    windows.append([x, y, x + bw, y + bh])
        return windows

    windows = sliding_windows(w, h, WINDOW_SCALES, WINDOW_STRIDE)

    # Feature extraction
    def extract_hog_feature(patch_bgr):
        gray = cv2.cvtColor(patch_bgr, cv2.COLOR_BGR2GRAY)
        gray_normalized = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
        hog_vec = HOG.compute(gray_normalized).reshape(-1)
        return hog_vec.astype(np.float32)

    feats = []
    valid_boxes = []

    for box in windows:
        x1, y1, x2, y2 = box
        patch = image_bgr[y1:y2, x1:x2]
        if patch.size == 0:
            continue
        patch_resized = cv2.resize(patch, PATCH_SIZE)
        feats.append(extract_hog_feature(patch_resized))
        valid_boxes.append(box)

    if len(feats) == 0:
        return []

    # Predict with SVM
    X = scaler.transform(np.asarray(feats, dtype=np.float32))
    scores_mat = clf.decision_function(X)

    raw_boxes, raw_scores, raw_labels = [], [], []
    for idx, row in enumerate(scores_mat):
        cls_id = int(np.argmax(row))
        score = float(row[cls_id])

        if cls_id < len(CLASSES) and score >= SCORE_THRESHOLD:
            raw_boxes.append(valid_boxes[idx])
            raw_scores.append(score)
            raw_labels.append(cls_id)

    if len(raw_boxes) == 0:
        return []

    # Apply NMS
    keep_indices = nms_boxes(raw_boxes, raw_scores, NMS_IOU_THRESHOLD)

    detections = []
    for idx in keep_indices:
        box = raw_boxes[idx]
        score = raw_scores[idx]
        label = raw_labels[idx]

        detections.append({
            'bbox': [float(box[0]), float(box[1]), float(box[2]), float(box[3])],
            'class_id': label,
            'class_name': CLASSES[label],
            'confidence': score
        })

    return detections

print("HOG+SVM prediction function defined.")

HOG+SVM prediction function defined.


In [9]:
# ============================================================================
# SECTION 4: Visualization & Results Processing
# ============================================================================

def draw_detections(image_rgb, detections, thickness=2, font_scale=0.6):
    """
    Draw bounding boxes and labels on image.

    Args:
        image_rgb: RGB numpy array
        detections: List of detection dicts with 'bbox', 'class_name', 'confidence'
        thickness: Box line thickness
        font_scale: Text font scale

    Returns:
        Annotated RGB image
    """
    image = image_rgb.copy()

    colors = {
        'Bus': (0, 255, 0),        # Green
        'Car': (255, 0, 0),        # Blue (BGR)
        'Motorbike': (0, 165, 255),  # Orange
        'Truck': (128, 0, 128)     # Purple
    }

    for det in detections:
        x1, y1, x2, y2 = [int(v) for v in det['bbox']]
        class_name = det['class_name']
        confidence = det['confidence']

        # Draw box
        color = colors.get(class_name, (255, 255, 255))
        cv2.rectangle(image, (x1, y1), (x2, y2), color, thickness)

        # Draw label
        label = f"{class_name} {confidence:.2f}"
        text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1)[0]
        cv2.rectangle(image, (x1, y1 - text_size[1] - 5), (x1 + text_size[0], y1), color, -1)
        cv2.putText(image, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX,
                   font_scale, (255, 255, 255), 1, cv2.LINE_AA)

    return image

def get_detection_counts(detections):
    """Count detections by class."""
    counts = defaultdict(int)
    for det in detections:
        counts[det['class_name']] += 1
    return dict(counts)

print("Visualization functions defined.")

Visualization functions defined.


In [10]:
# ============================================================================
# SECTION 5: Model Manager Class
# ============================================================================

class VehicleDetectionInference:
    """Unified inference interface for all vehicle detection models."""

    def __init__(self):
        self.models = {}
        self.load_all_models()

    def load_all_models(self):
        """Load all available models."""
        try:
            self.models['yolo'] = load_yolo_model()
            print("✓ YOLOv8 loaded")
        except Exception as e:
            print(f"✗ Failed to load YOLOv8: {e}")

        try:
            self.models['ssd'] = load_ssd_model()
            print("✓ SSD300 loaded")
        except Exception as e:
            print(f"✗ Failed to load SSD300: {e}")

        try:
            self.models['fasterrcnn'] = load_fasterrcnn_model()
            print("✓ FasterRCNN loaded")
        except Exception as e:
            print(f"✗ Failed to load FasterRCNN: {e}")

        try:
            self.models['hog_svm'] = load_hog_svm_model()
            print("✓ HOG+SVM loaded")
        except Exception as e:
            print(f"✗ Failed to load HOG+SVM: {e}")

    def predict(self, image_rgb, model_name):
        """
        Run inference on image with specified model.

        Args:
            image_rgb: RGB numpy array (H x W x 3)
            model_name: 'yolo', 'ssd', 'fasterrcnn', or 'hog_svm'

        Returns:
            Tuple of (detections, inference_time_ms)
        """
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} not loaded. Available: {list(self.models.keys())}")

        start_time = time.time()

        if model_name == 'yolo':
            detections = predict_yolo(self.models['yolo'], image_rgb)
        elif model_name == 'ssd':
            detections = predict_ssd(self.models['ssd'], image_rgb)
        elif model_name == 'fasterrcnn':
            detections = predict_fasterrcnn(self.models['fasterrcnn'], image_rgb)
        elif model_name == 'hog_svm':
            detections = predict_hog_svm(self.models['hog_svm'], image_rgb)

        elapsed_ms = (time.time() - start_time) * 1000
        return detections, elapsed_ms

    def get_available_models(self):
        """Return list of successfully loaded models."""
        return list(self.models.keys())

print("VehicleDetectionInference class defined.")

VehicleDetectionInference class defined.


In [11]:
# ============================================================================
# SECTION 6: Initialize Inference Pipeline
# ============================================================================

# Create unified inference manager
print("Initializing Vehicle Detection Inference Pipeline...")
inference_engine = VehicleDetectionInference()
available_models = inference_engine.get_available_models()

print(f"\n✓ Pipeline Ready!")
print(f"Available models: {available_models}")

Initializing Vehicle Detection Inference Pipeline...
Loading YOLOv8 model...
✓ YOLOv8 loaded
Loading SSD300 model...
Downloading: "https://download.pytorch.org/models/vgg16_features-amdegroot-88682ab5.pth" to /root/.cache/torch/hub/checkpoints/vgg16_features-amdegroot-88682ab5.pth


100%|██████████| 528M/528M [00:08<00:00, 66.5MB/s]


✓ SSD300 loaded
Loading FasterRCNN model...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 78.9MB/s]


✓ FasterRCNN loaded
Loading HOG+SVM model...
✓ HOG+SVM loaded

✓ Pipeline Ready!
Available models: ['yolo', 'ssd', 'fasterrcnn', 'hog_svm']


In [12]:
# ============================================================================
# SECTION 7: Example Inference on Test Image
# ============================================================================

def demo_single_inference(image_path, model_name='yolo'):
    """
    Run inference on a single image.

    Args:
        image_path: Path to image file
        model_name: Model to use ('yolo', 'ssd', 'fasterrcnn', 'hog_svm')

    Returns:
        Tuple of (annotated_image, detections_dict)
    """
    # Load image
    image_rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

    # Run inference
    print(f"\nRunning {model_name.upper()} inference...")
    detections, inference_time = inference_engine.predict(image_rgb, model_name)

    # Annotate
    annotated = draw_detections(image_rgb, detections)
    counts = get_detection_counts(detections)

    # Print results
    print(f"Inference time: {inference_time:.2f} ms")
    print(f"Detections found: {len(detections)}")
    print(f"Class breakdown: {counts}")

    return annotated, {
        'detections': detections,
        'inference_time_ms': inference_time,
        'counts': counts
    }

print("Demo inference function defined.")

Demo inference function defined.


In [13]:
# ============================================================================
# SECTION 8: Interactive Gradio Interface
# ============================================================================

def predict_gradio(image, model_name):
    """
    Gradio callback for interactive inference.

    Args:
        image: PIL Image from Gradio
        model_name: Selected model name

    Returns:
        Tuple of (annotated_image_pil, metrics_text)
    """
    if image is None:
        return None, "Please upload an image"

    try:
        # Convert PIL to numpy RGB
        image_rgb = np.array(image).astype(np.uint8)

        # Handle grayscale images
        if len(image_rgb.shape) == 2:
            image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_GRAY2RGB)
        elif image_rgb.shape[2] == 4:  # RGBA
            image_rgb = cv2.cvtColor(image_rgb, cv2.COLOR_RGBA2RGB)

        # Run inference
        detections, inference_time = inference_engine.predict(image_rgb, model_name)

        # Annotate
        annotated_rgb = draw_detections(image_rgb, detections)
        annotated_pil = Image.fromarray(annotated_rgb)

        # Prepare metrics text
        counts = get_detection_counts(detections)
        counts_str = "\n".join([f"{cls}: {count}" for cls, count in sorted(counts.items())])

        metrics = f"""
Model: {model_name.upper()}
Inference Time: {inference_time:.2f} ms
Total Detections: {len(detections)}

Detections by Class:
{counts_str if counts_str else "No vehicles detected"}
        """.strip()

        return annotated_pil, metrics

    except Exception as e:
        return None, f"Error: {str(e)}"

# Create Gradio interface
demo = gr.Interface(
    fn=predict_gradio,
    inputs=[
        gr.Image(type='pil', label='Upload Vehicle Image'),
        gr.Dropdown(
            choices=available_models,
            value=available_models[0] if available_models else 'yolo',
            label='Select Detection Model'
        )
    ],
    outputs=[
        gr.Image(label='Detection Results', type='pil'),
        gr.Textbox(label='Detection Metrics', lines=8)
    ],
    title='🚗 Vehicle Detection Comparison',
    description='''
Unified inference interface for vehicle detection models.
Compare YOLOv8, SSD300, FasterRCNN, and HOG+SVM on the same image.
    ''',
    examples=None,
    allow_flagging='never',
    theme=gr.themes.Soft()
)

print("✓ Gradio interface created")
print("\nTo launch the interface, run: demo.launch(share=False)")

/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


✓ Gradio interface created

To launch the interface, run: demo.launch(share=False)


In [ ]:
# ============================================================================
# SECTION 9: Launch Interactive Demo
# ============================================================================

# Uncomment below to launch Gradio interface
demo.launch(debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d91ab050e345f6bb4a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Usage Guide & Technical Documentation

## Unified Inference Pipeline

This deployment notebook provides a standardized API for all 4 vehicle detection models.

### Model Characteristics

| Model | Type | Speed | Accuracy | Memory | Best For |
|-------|------|-------|----------|--------|----------|
| **YOLOv8** | Single-stage detector | ⚡⚡⚡ Fast | ✓✓✓ High | Low | Real-time applications |
| **SSD300** | Single-shot detector | ⚡⚡ Medium | ✓✓ Medium | Medium | Balanced speed/accuracy |
| **FasterRCNN** | Two-stage detector | ⚡ Slow | ✓✓✓ High | High | Highest accuracy needed |
| **HOG+SVM** | Sliding window (classical) | ⚡ Slow | ✓ Lower | Very Low | Resource-constrained environments |

### Unified Detection Format

All models return detections in a standardized dictionary format:
```python
detection = {
    'bbox': [x1, y1, x2, y2],        # Bounding box coordinates in original image space
    'class_id': 0,                    # Class ID (0-3 for Bus, Car, Motorbike, Truck)
    'class_name': 'Car',              # Human-readable class name
    'confidence': 0.95                # Detection confidence (0.0-1.0)
}
```

### Quick Start Examples

```python
# Initialize engine
engine = VehicleDetectionInference()

# Load image
import cv2
image_rgb = cv2.cvtColor(cv2.imread('test.jpg'), cv2.COLOR_BGR2RGB)

# Single model inference
detections, time_ms = engine.predict(image_rgb, 'yolo')
print(f"Found {len(detections)} vehicles in {time_ms:.2f}ms")

# Compare all models
for model in engine.get_available_models():
    dets, t = engine.predict(image_rgb, model)
    print(f"{model}: {len(dets)} detections in {t:.2f}ms")
```

In [ ]:
# ============================================================================
# SECTION 10: Batch Processing & Analysis
# ============================================================================

def batch_predict(image_dir, model_names=['yolo', 'ssd', 'fasterrcnn', 'hog_svm'],
                 output_dir=None):
    """
    Run batch inference on all images in a directory.

    Args:
        image_dir: Path to directory containing images
        model_names: List of models to use
        output_dir: Optional directory to save annotated images

    Returns:
        Dictionary with statistics
    """
    import glob

    image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')) +
                        glob.glob(os.path.join(image_dir, '*.png')))

    if not image_paths:
        print(f"No images found in {image_dir}")
        return {}

    results = {model: {'times': [], 'det_counts': []} for model in model_names}

    for img_path in image_paths:
        image_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        for model_name in model_names:
            try:
                detections, time_ms = inference_engine.predict(image_rgb, model_name)
                results[model_name]['times'].append(time_ms)
                results[model_name]['det_counts'].append(len(detections))

                if output_dir:
                    annotated = draw_detections(image_rgb, detections)
                    out_path = os.path.join(output_dir,
                                           f"{Path(img_path).stem}_{model_name}.jpg")
                    cv2.imwrite(out_path, cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))

            except Exception as e:
                print(f"Error processing {img_path} with {model_name}: {e}")

    # Compute statistics
    stats = {}
    for model_name in model_names:
        times = results[model_name]['times']
        counts = results[model_name]['det_counts']

        if times:
            stats[model_name] = {
                'avg_time_ms': np.mean(times),
                'min_time_ms': np.min(times),
                'max_time_ms': np.max(times),
                'fps': 1000 / np.mean(times),
                'total_detections': sum(counts),
                'avg_detections_per_image': np.mean(counts)
            }

    return stats

print("Batch processing functions defined.")

# Inference Architecture & Data Flow

## Processing Pipeline

Each model follows this standardized pipeline:

### 1. Input Loading
- Accept RGB numpy array (H × W × 3)
- Support PIL Images via Gradio

### 2. Model-Specific Preprocessing
- **YOLOv8**: Direct BGR conversion, handled internally
- **SSD300**: Resize to 300×300, normalize
- **FasterRCNN**: Resize to 640×640, normalize to [0, 1]
- **HOG+SVM**: Sliding window generation, HOG descriptor extraction

### 3. Inference
- **YOLOv8**: Single pass through network
- **SSD/FasterRCNN**: PyTorch models on GPU/CPU
- **HOG+SVM**: SVM decision function across all windows

### 4. Output Processing & Filtering
- Apply confidence threshold (CONF_THRESHOLD = 0.5)
- Scale coordinates back to original image space
- Convert class IDs to human-readable names
- Apply NMS (for HOG+SVM)

### 5. Visualization
- Draw colored bounding boxes per class
- Render class name and confidence
- Return annotated RGB image

## Key Differences Between Models

### YOLOv8
- **Advantages**: Fastest, real-time capable, high accuracy
- **Process**: End-to-end detection, no post-processing
- **Input**: BGR image (any size)
- **Output**: 0-indexed class IDs

### SSD300
- **Advantages**: Fast, moderate accuracy
- **Process**: Single shot, multi-scale feature maps
- **Input**: Fixed 300×300 RGB
- **Output**: 1-indexed class IDs (must convert)

### FasterRCNN
- **Advantages**: Highest accuracy, better for small objects
- **Process**: Two-stage (RPN + ROI pooling)
- **Input**: Fixed 640×640 RGB
- **Output**: 1-indexed class IDs (must convert)

### HOG+SVM
- **Advantages**: Explainable, minimal dependencies
- **Process**: Multi-scale sliding window + hand-crafted features
- **Input**: RGB image (any size)
- **Output**: 0-indexed class IDs
- **Postprocessing**: Manual NMS required

In [ ]:
# ============================================================================
# SECTION 11: Validation & Testing
# ============================================================================

def validate_inference():
    """
    Validate that all models produce correctly formatted output.

    Returns:
        Dictionary with validation results
    """
    print("=" * 60)
    print("VALIDATING INFERENCE PIPELINE")
    print("=" * 60)

    # Create synthetic test image
    test_image = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)

    validation_results = {}

    for model_name in inference_engine.get_available_models():
        print(f"\nTesting {model_name.upper()}...")

        try:
            detections, time_ms = inference_engine.predict(test_image, model_name)

            # Validate output format
            valid = True
            issues = []

            # Check list of dicts
            if not isinstance(detections, list):
                valid = False
                issues.append(f"Output is {type(detections)}, expected list")

            # Check each detection
            for det in detections:
                if not isinstance(det, dict):
                    valid = False
                    issues.append(f"Detection is {type(det)}, expected dict")
                    continue

                required_keys = {'bbox', 'class_id', 'class_name', 'confidence'}
                if not required_keys.issubset(det.keys()):
                    valid = False
                    issues.append(f"Missing keys: {required_keys - set(det.keys())}")

                # Validate bbox format
                if 'bbox' in det:
                    bbox = det['bbox']
                    if not isinstance(bbox, (list, tuple)) or len(bbox) != 4:
                        valid = False
                        issues.append(f"bbox should be [x1, y1, x2, y2], got {bbox}")

                    # Check coordinates are within image bounds (with small margin)
                    if bbox[0] < -10 or bbox[2] > 640 + 10:
                        issues.append(f"bbox x-coordinates out of bounds: {bbox}")
                    if bbox[1] < -10 or bbox[3] > 480 + 10:
                        issues.append(f"bbox y-coordinates out of bounds: {bbox}")

                # Validate class_id
                if 'class_id' in det:
                    if not (0 <= det['class_id'] < len(CLASSES)):
                        valid = False
                        issues.append(f"class_id {det['class_id']} out of range [0, {len(CLASSES)-1}]")

                # Validate confidence
                if 'confidence' in det:
                    if not (0.0 <= det['confidence'] <= 1.0):
                        valid = False
                        issues.append(f"confidence {det['confidence']} not in [0.0, 1.0]")

            # Print results
            status = "✓ PASS" if valid else "✗ FAIL"
            print(f"  Status: {status}")
            print(f"  Inference time: {time_ms:.2f} ms")
            print(f"  Detections: {len(detections)}")

            if issues:
                print("  Issues found:")
                for issue in issues[:3]:  # Show first 3 issues
                    print(f"    - {issue}")

            validation_results[model_name] = {
                'valid': valid,
                'inference_time_ms': time_ms,
                'num_detections': len(detections),
                'issues': issues
            }

        except Exception as e:
            print(f"  ✗ ERROR: {str(e)}")
            validation_results[model_name] = {
                'valid': False,
                'error': str(e)
            }

    print("\n" + "=" * 60)
    all_valid = all(r['valid'] for r in validation_results.values() if 'valid' in r)
    print(f"Overall: {'✓ ALL TESTS PASSED' if all_valid else '✗ SOME TESTS FAILED'}")
    print("=" * 60)

    return validation_results

# Run validation
print("Validation function defined. Run: validate_inference()")